In [1]:
#chargement de la base de donnees 
from datasets import load_dataset
dataset=load_dataset("stanfordnlp/imdb")
print(dataset['train']['text'][0])
print("label (0=negatif, 1=positif): ", dataset['train']['label'][0])

/Users/valeriiarybalka/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [2]:
import pandas as pd

# On extrait les 5 premières lignes du dataset d'entraînement
apercu = dataset['train'][0:5]

# On les transforme en tableau Pandas (DataFrame)
df_apercu = pd.DataFrame(apercu)

df_apercu

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


L'Intelligence Artificielle ne sait pas lire des mots comme "chef-d'œuvre" ou "ennuyeux". Elle ne comprend que les mathématiques. Nous allons donc utiliser TF-IDF.

Qu'est-ce que TF-IDF ?
C'est une technique qui va regarder les 25 000 critiques et créer un dictionnaire géant. Pour chaque mot, elle va calculer un score :

Un mot très courant comme "le" ou "et" aura un score proche de 0 (car il n'aide pas à deviner le sentiment).

Un mot fort comme "masterpiece" (chef-d'œuvre) ou "awful" (affreux) aura un score élevé.

TF (Term Frequency = Fréquence du terme) : On compte combien de fois un mot apparaît dans le texte analysé. Exemple : Dans une critique du film Inception, le mot "rêve" apparaît 5 fois. Son score TF est donc élevé.

IDF (Inverse Document Frequency = Fréquence inverse) : On regarde si ce mot est rare ou commun dans toutes les autres critiques de la base de données. Le mot "film" apparaît dans 100 % des critiques. Son score IDF sera de 0. Le mot "rêve" apparaît seulement dans 1 % des critiques. Son score IDF sera très élevé.

Le score final d'un mot est la multiplication des deux indicateurs.
Pour "film" : TF élevé × IDF nul = Score de 0. L'algorithme l'ignore.
Pour "rêve" : TF élevé × IDF élevé = Score énorme. L'algorithme comprend que c'est un mot clé distinctif de ce texte.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train=dataset['train']['text']
X_test=dataset['test']['text']

y_train=dataset['train']['label']
y_test=dataset['test']['label']

vectorizer=TfidfVectorizer(max_features=10000)
print("traduiction de texte en math en cours ...")
X_train_vec=vectorizer.fit_transform(X_train)
X_test_vec=vectorizer.transform(X_test)
print("traduction terminee!")
print("taille de nouvelle matrice mathematique :", X_train_vec.shape)

traduiction de texte en math en cours ...
traduction terminee!
taille de nouvelle matrice mathematique : (25000, 10000)


In [4]:
import tensorflow as tf
print(tf.__version__)

2.16.2


In [5]:
#svm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.losses import Hinge
from tensorflow.keras.layers import Input
import numpy as np


In [9]:
y_train_array=np.array(y_train)
y_test_array=np.array(y_test)
y_train1=np.where(y_train_array==0,-1,1)
y_test1=np.where(y_test_array==0,-1,1)

model=Sequential()
model.add(Input(shape=(X_train_vec.shape[1],)))
model.add(Dense(1))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 1)              │        10,001 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,001 (39.07 KB)

 Trainable params: 10,001 (39.07 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
print("Forme de X_train_vec :", X_train_vec.shape)
print("Forme de y_train1 :", y_train1.shape)

Forme de X_train_vec : (25000, 10000)
Forme de y_train1 : (25000,)


In [11]:
model.compile(optimizer="adam", loss=Hinge(), metrics=['accuracy'])
history=model.fit(X_train_vec.toarray(), y_train1, batch_size=64, epochs=10)
 

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 386us/step - accuracy: 0.0000e+00 - loss: 0.9580
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 366us/step - accuracy: 0.0055 - loss: 0.7759  
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 425us/step - accuracy: 0.1596 - loss: 0.6184
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 407us/step - accuracy: 0.2754 - loss: 0.5121
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 412us/step - accuracy: 0.3243 - loss: 0.4466
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 406us/step - accuracy: 0.3415 - loss: 0.4046
Epoch 7/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 406us/step - accuracy: 0.3578 - loss: 0.3727
Epoch 8/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 376us/step - accuracy: 0.3622 - loss: 0.3453
Epoch 9/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 368us/step - accuracy: 0.3743 - loss: 0.3252
Epoch 10/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 370us/step - accuracy: 0.3846 - loss: 0.3048


In [13]:
#test avec .evaluate
res=model.evaluate(X_test_vec.toarray(), y_test1, verbose=0)
print(f"--- Résultat de .evaluate() ---")
print("test loss, test acc(fausse)", res)

#test avec .predict
pred=model.predict(X_test_vec.toarray())
pred_classes=np.where(pred>0,1,-1)
acc=np.mean(pred_classes.flatten()==y_test1)
print(f"--- Résultat de .predict() ---")
print(f"Vraie Accuracy du SVM (basée sur le signe) : {acc * 100:.2f}%")

--- Résultat de .evaluate() ---
test loss, test acc(fausse) [0.3704589307308197, 0.34088000655174255]
782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 214us/step
--- Résultat de .predict() ---
Vraie Accuracy du SVM (basée sur le signe) : 88.05%


In [15]:
phrases_test = [
    # Positives claires
    "An absolute masterpiece from start to finish, breathtaking cinematography.",
    "I loved every single minute of this film, the characters were so deep.",
    "Brilliant acting and a truly engaging plot.",
    
    # Négatives claires
    "A total waste of time and money. Absolutely terrible.",
    "The worst movie I have seen this year, completely unwatchable.",
    "Horrible script, awful direction, zero redeeming qualities.",
    
    # Nuancées / Mixtes (pour tester les limites du modèle linéaire)
    "The beginning was exciting, but the ending made no sense at all.",
    "Not the best movie ever, but it had some charming moments.",
    "Visually stunning, yet the story felt utterly empty and boring."
]

# Boucle rapide pour tester toutes les phrases d'un coup
for phrase in phrases_test:
    # 1. Vectorisation de la phrase avec votre TfidfVectorizer existant
    vec = vectorizer.transform([phrase])
    # 2. Prédiction brute du modèle
    score = model.predict(vec.toarray())[0][0]
    # 3. Attribution de la classe selon le signe (> 0 : Positif, < 0 : Négatif)
    label = "Positif 🟢" if score > 0 else "Négatif 🔴"
    print(f'"{phrase}" => Score: {score:.2f} => {label}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
"An absolute masterpiece from start to finish, breathtaking cinematography." => Score: 0.79 => Positif 🟢
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
"I loved every single minute of this film, the characters were so deep." => Score: 0.33 => Positif 🟢
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
"Brilliant acting and a truly engaging plot." => Score: 0.41 => Positif 🟢
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
"A total waste of time and money. Absolutely terrible." => Score: -2.50 => Négatif 🔴
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
"The worst movie I have seen this year, completely unwatchable." => Score: -1.68 => Négatif 🔴
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
"Horrible script, awful direction, zero redeeming qualities." => Score: -2.68 => Négatif 🔴
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
"The beginning was exciting, but the ending made no sense at all." => Score: -0.38 => Négatif 🔴
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
"Not the best movie ever, but it had some charming momen

In [18]:
print("Nombre de couches :", len(model.layers))
for i, layer in enumerate(model.layers):
    print(f"Couche {i} : {type(layer)}")
model.layers[0].get_weights()

Nombre de couches : 1
Couche 0 : <class 'keras.src.layers.core.dense.Dense'>


[array([[-0.19639969],
        [-0.2832393 ],
        [ 0.74001485],
        ...,
        [-0.16115095],
        [ 0.37428862],
        [ 0.4455154 ]], dtype=float32),
 array([0.13259803], dtype=float32)]

In [21]:
indices=model.layers[0].get_weights()[0].flatten().argsort()
mots=vectorizer.get_feature_names_out()
print("les mots les plus negatifs", mots[indices[:10]])
print("les mots les plus positifs", mots[indices[-10:]])

les mots les plus negatifs ['worst' 'awful' 'poor' 'bad' 'boring' 'waste' 'nothing' 'dull' 'terrible'
 'worse']
les mots les plus positifs ['enjoyed' 'well' 'favorite' 'loved' 'wonderful' 'best' 'perfect'
 'amazing' 'excellent' 'great']
